In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab6MlApp") \
    .getOrCreate()

sc = spark.sparkContext

Charger les données cancer_reg (un header est présent, de plus, utiliser l'option inferSchema).

In [ ]:
# Votre code ici

A l'aide de la commande dir, afficher les méthodes pouvant être utilisées sur data. Afficher les 10 premières lignes du dataframe ainsi chargé.

In [ ]:
# Votre code ici

Supprimer du dataframe les colonnes binnedInc, Geography, PctSomeCol18_24, PctEmployed16_Over et PctPrivateCoverageAlone.

Repérer la colonne cible (target) et stocker des variables la liste des noms des colonnes des variables explicatives d'une part et le nom de la colonne cible d'autre part. 

In [ ]:
# Votre code ici

Séparer les données en un jeu d'entrainement et un jeu de test à l'aide de la méthode randomSplit. Choisir une répartition 80%-20%.

In [ ]:
# Votre code ici

LEVER LA MAIN SUR TEAMS

In [2]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

Assembler les features dans un vecteur colonne unique à l'aide de VectorAssembler.

In [ ]:
# Votre code ici

Instancier un modèle de régression linéaire en spécifiant les colonnes des variables explicatives et la colonne labellisée.

In [ ]:
# Votre code ici

Créer une pipeline contenant les 2 étapes de l'assembleur et du modèle de régression linéaire.

Entraîner la pipeline sur le jeu d'entraînement.

In [ ]:
# Votre code ici

Réaliser les prédictions sur le jeu de test. Afficher les 10 premières lignes avec leurs prédictions.

In [ ]:
# Votre code ici

Créer un évaluateur correspondant au problème donné (en identifiant bien la typologie du problème). L'utiliser pour évaluer le rmse du modèle obtenu.

In [3]:
path_file="../../data/cancer_reg.csv"
df_csv=spark.read.csv(path_file,header=True, inferSchema=True)
df_csv.printSchema()
df_csv.show()

print(dir(df_csv))

df_csv.show(2, vertical=True)

df_csv_net=df_csv.drop("binnedInc","Geography","PctSomeCol18_24","PctEmployed16_Over","PctPrivateCoverageAlone")

col_targ = "TARGET_deathRate"
col_expl=[c for c in df_csv_net.columns if c != col_targ]

train_df, test_df = df_csv_net.randomSplit([0.8, 0.2], seed=1)
train_df.count(), test_df.count()

assembler = VectorAssembler(
    inputCols=col_expl,
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol=col_targ,
    predictionCol="prediction"
)

pipeline = Pipeline(stages=[assembler, lr])

model_pipeline = pipeline.fit(train_df)

predictions = model_pipeline.transform(test_df)
predictions.select("prediction", col_targ, "features").show(10)
predictions.toPandas()

evaluator = RegressionEvaluator(
    labelCol=col_targ,       
    predictionCol="prediction", 
    metricName="rmse"       
)
rmse = evaluator.evaluate(predictions)
print(rmse)

evaluator_r2 = RegressionEvaluator(
    labelCol=col_targ,       
    predictionCol="prediction", 
    metricName="r2"       
)
r2 = evaluator_r2.evaluate(predictions)
print(r2)

evaluator = RegressionEvaluator(
    labelCol=col_targ,       
    predictionCol="prediction"     
)
rmse = evaluator.evaluate(predictions,{evaluator.metricName:'rmse'})
r2 = evaluator.evaluate(predictions,{evaluator.metricName:'r2'})
rmse,r2

root
 |-- avgAnnCount: double (nullable = true)
 |-- avgDeathsPerYear: integer (nullable = true)
 |-- TARGET_deathRate: double (nullable = true)
 |-- incidenceRate: double (nullable = true)
 |-- medIncome: integer (nullable = true)
 |-- popEst2015: integer (nullable = true)
 |-- povertyPercent: double (nullable = true)
 |-- studyPerCap: double (nullable = true)
 |-- binnedInc: string (nullable = true)
 |-- MedianAge: double (nullable = true)
 |-- MedianAgeMale: double (nullable = true)
 |-- MedianAgeFemale: double (nullable = true)
 |-- Geography: string (nullable = true)
 |-- AvgHouseholdSize: double (nullable = true)
 |-- PercentMarried: double (nullable = true)
 |-- PctNoHS18_24: double (nullable = true)
 |-- PctHS18_24: double (nullable = true)
 |-- PctSomeCol18_24: double (nullable = true)
 |-- PctBachDeg18_24: double (nullable = true)
 |-- PctHS25_Over: double (nullable = true)
 |-- PctBachDeg25_Over: double (nullable = true)
 |-- PctEmployed16_Over: double (nullable = true)
 |--

(19.859172391426515, 0.5292225158740396)

Reprendre les questions précédentes en rajoutant une étape d'ACP à 10 composantes entre le preprocessing et le modèle.

In [4]:
from pyspark.ml.feature import PCA

In [6]:
assembler = VectorAssembler(
    inputCols=col_expl,
    outputCol="features"
)

pca = PCA(
    k=10,
    inputCol="features",
    outputCol="pca_features"
)

lr_2 = LinearRegression(
    featuresCol="pca_features",
    labelCol=col_targ,
    predictionCol="prediction"
)

pipeline_2 = Pipeline(stages=[assembler, pca, lr_2])

model_pipeline_2 = pipeline_2.fit(train_df)

predictions_2 = model_pipeline_2.transform(test_df)
predictions_2.select("prediction", col_targ, "features", "pca_features").show(10)

evaluator = RegressionEvaluator(
    labelCol=col_targ,       
    predictionCol="prediction"     
)
rmse = evaluator.evaluate(predictions_2,{evaluator.metricName:'rmse'})
r2 = evaluator.evaluate(predictions_2,{evaluator.metricName:'r2'})
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

+------------------+----------------+--------------------+--------------------+
|        prediction|TARGET_deathRate|            features|        pca_features|
+------------------+----------------+--------------------+--------------------+
|112.47995958694146|           203.3|[8.0,3.0,201.3,68...|[-6298.4487925559...|
|141.78687635832654|           166.4|[9.0,4.0,347.2,56...|[-2806.5754029224...|
|164.83949953163216|           172.6|[9.0,5.0,355.4,42...|[-1783.2333063803...|
| 164.9172209572051|           192.3|[9.0,5.0,342.6,32...|[-1711.4761637035...|
|144.40020559903724|           117.6|[10.0,4.0,341.9,5...|[-2474.2902687558...|
|188.22675590679688|           187.3|[11.0,4.0,502.1,4...|[-1782.7812952589...|
| 137.7735584609302|           106.1|[12.0,4.0,318.9,6...|[-4099.4773413825...|
|150.02380342439588|           129.3|[12.0,4.0,366.9,4...|[-3061.5718806710...|
|156.14261601859323|           104.8|[13.0,4.0,336.1,4...|[-2826.3481484962...|
| 167.5306339240832|           126.3|[13

Conclure quant à l'utilité de l'ACP dans ce problème à dimension raisonnable.

Arrêter la session Spark.

In [ ]:
spark.stop()